<a href="https://colab.research.google.com/github/SaymaSJ/3-DOF-Cobot--Dissertation/blob/main/yolov8_and_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from ultralytics import YOLO
import cv2, time, os, csv, glob

IMG_DIR = "data/val/images"
OUT_CSV = "det_metrics.csv"
MODELS = {
  "yolov8": "weights/yolov8n_bread.pt",   # fallback: "yolov8n.pt"
  "yolov10": "weights/yolov10n_bread.pt"
}
CONF, IOU = 0.35, 0.5
IMGS = sorted(glob.glob(os.path.join(IMG_DIR, "*.jpg")))

with open(OUT_CSV, "w", newline="") as f:
    w = csv.writer(f); w.writerow(["model","image","latency_ms","detections"])
    for name, path in MODELS.items():
        m = YOLO(path)
        # warmup
        for _ in range(3): m.predict(IMGS[0], conf=CONF, iou=IOU, verbose=False)
        for img in IMGS:
            t0 = time.time()
            r = m.predict(img, conf=CONF, iou=IOU, verbose=False)[0]
            dt = (time.time()-t0)*1000.0
            dets = len(r.boxes)
            w.writerow([name, os.path.basename(img), f"{dt:.1f}", dets])

print("Wrote det_metrics.csv")


ModuleNotFoundError: No module named 'ultralytics'

In [ ]:
!pip install ultralytics
# For YOLOv10, additional installation may be needed (see below)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.0 MB/s eta 0:00:00


In [ ]:
!pip install git+https://github.com/THU-MIG/yolov10.git

  Cloning https://github.com/THU-MIG/yolov10.git to /tmp/pip-req-build-cren8uc0
  Running command git clone --filter=blob:none --quiet https://github.com/THU-MIG/yolov10.git /tmp/pip-req-build-cren8uc0
  Resolved https://github.com/THU-MIG/yolov10.git to commit 453c6e38a51e9d1d5a2aa5fb7f1014a711913397
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.1.34-py3-none-any.whl size=732493 sha256=185941391570be74e634c4ed95ac3ab972370a0807fac412c8fd7d8824253041
  Stored in directory: /tmp/pip-ephem-wheel-cache-a33v2ntc/wheels/35/9a/d9/ec58d6096ceff21c4a77b18cd266af11385b347ca19a307597
Successfully built ultralytics
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.3.202
    Uninstalling ultralytics-8.3.202:
      Successfully uninstalled ultralytics-8.3.202


In [ ]:
!pip install ultralytics easyocr pytesseract opencv-python pillow numpy
!apt-get install tesseract-ocr


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 963.8/963.8 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.1/292.1 kB 19.0 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.


In [ ]:
#!/usr/bin/env python3
"""
Detect expiry dates from video using YOLOv10 + ROI OCR.
- Input: video file or webcam (e.g., 0)
- Output: CSV with per-frame predictions, optional annotated MP4

Dependencies (example):
  pip install ultralytics==8.3.0 easyocr==1.7.1 pytesseract pillow numpy opencv-python
  # (Linux) also: sudo apt-get install tesseract-ocr

Usage examples:
  python detect_expiry_yolov10.py --model weights/yolov10n_bread.pt --source data/video.mp4 \
      --out_csv expiry_log.csv --save_video out_annotated.mp4 --conf 0.35 --iou 0.5

  # webcam
  python detect_expiry_yolov10.py --model weights/yolov10n_bread.pt --source 0 --show
"""

import argparse, csv, os, re, sys, time
from typing import Optional, Tuple

import cv2
import numpy as np
from ultralytics import YOLO

# --- OCR backends (EasyOCR primary, pytesseract fallback) ---
_USE_EASYOCR = True
try:
    import easyocr
except Exception:
    _USE_EASYOCR = False

try:
    import pytesseract
except Exception:
    pytesseract = None


def parse_args():
    p = argparse.ArgumentParser(description="YOLOv10 + ROI OCR for expiry date")
    p.add_argument("--model", required=True, type=str, help="Path to YOLOv10 .pt weights")
    p.add_argument("--source", required=True, type=str,
                   help="Video path or webcam index (e.g., 0)")
    p.add_argument("--out_csv", type=str, default="expiry_log.csv",
                   help="CSV path to write frame-wise predictions")
    p.add_argument("--save_video", type=str, default="",
                   help="Optional: output annotated video path (e.g., out.mp4)")
    p.add_argument("--conf", type=float, default=0.35, help="YOLO confidence")
    p.add_argument("--iou", type=float, default=0.5, help="YOLO IoU")
    p.add_argument("--target_class", type=int, default=-1,
                   help="If set (>=0), only use boxes of this class id as ROI")
    p.add_argument("--show", action="store_true", help="Show live window")
    p.add_argument("--max_frames", type=int, default=0, help="Stop after N frames (0 = all)")
    p.add_argument("--gpu", action="store_true", help="Try GPU for EasyOCR if available")
    return p.parse_args()


# --- date parsing & normalization ---
DATE_RE = re.compile(r"(\d{4}[\/\-]\d{2}[\/\-]\d{2}|\d{2}[\/\-]\d{2}[\/\-]\d{2,4})")

def normalize_date(s: str) -> str:
    """
    Return YYYY-MM-DD if a date-like pattern exists, else "".
    - Accepts YYYY-MM-DD or DD-MM-YYYY (also with slashes). Two-digit year coerced to 20xx/19xx.
    """
    s = s.strip().replace("/", "-")
    m = DATE_RE.search(s)
    if not m:
        return ""
    tok = m.group(1)
    parts = tok.split("-")
    if len(parts[0]) == 4:
        # Already YYYY-MM-DD
        year, mo, day = parts
        return f"{year.zfill(4)}-{mo.zfill(2)}-{day.zfill(2)}"

    # Assume DD-MM-YYYY or DD-MM-YY
    day, mo, yr = parts
    if len(yr) == 2:
        yr2 = int(yr)
        yr = f"20{yr}" if yr2 <= 49 else f"19{yr}"
    return f"{yr.zfill(4)}-{mo.zfill(2)}-{day.zfill(2)}"


# --- OCR helpers ---
class OCR:
    def __init__(self, use_gpu=False):
        self.backend = None
        self.reader = None

        if _USE_EASYOCR:
            try:
                self.reader = easyocr.Reader(['en'], gpu=use_gpu)
                self.backend = "easyocr"
            except Exception as e:
                print(f"[WARN] EasyOCR init failed: {e}", file=sys.stderr)

        if self.reader is None and pytesseract is not None:
            self.backend = "tesseract"
        if self.backend is None:
            raise RuntimeError("No OCR backend available (EasyOCR and pytesseract missing).")

        print(f"[INFO] OCR backend: {self.backend}")

    def run(self, roi_bgr: np.ndarray) -> str:
        if roi_bgr.size == 0:
            return ""
        gray = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2GRAY)
        # binarize to help OCR; OTSU adapts thresholds
        thr = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]

        if self.backend == "easyocr":
            try:
                res = self.reader.readtext(thr)
                text = " ".join([t[1] for t in res]) if res else ""
                return text.strip()
            except Exception as e:
                print(f"[WARN] EasyOCR run failed: {e}", file=sys.stderr)
                return ""

        # pytesseract fallback
        if pytesseract is not None:
            try:
                config = "--psm 6"  # assume a block of text
                return pytesseract.image_to_string(thr, config=config).strip()
            except Exception as e:
                print(f"[WARN] pytesseract failed: {e}", file=sys.stderr)
                return ""

        return ""


# --- ROI selection from YOLO detections ---
def pick_roi_from_yolo(result, target_cls: int, frame_w: int, frame_h: int) -> Optional[Tuple[int,int,int,int]]:
    """
    Choose ROI from YOLO result:
      - If target_cls >= 0: pick the largest bbox with that class.
      - Else: pick the largest bbox overall.
    Returns (x1,y1,x2,y2) or None if no boxes.
    """
    boxes = getattr(result, "boxes", None)
    if boxes is None or len(boxes) == 0:
        return None

    candidates = []
    for b in boxes:
        xyxy = b.xyxy[0].tolist()
        x1, y1, x2, y2 = [int(v) for v in xyxy]
        cls_id = int(b.cls[0].item()) if b.cls is not None else -1
        if target_cls >= 0 and cls_id != target_cls:
            continue
        # clamp
        x1 = max(0, min(frame_w - 1, x1))
        x2 = max(0, min(frame_w - 1, x2))
        y1 = max(0, min(frame_h - 1, y1))
        y2 = max(0, min(frame_h - 1, y2))
        area = max(0, x2 - x1) * max(0, y2 - y1)
        if area > 0:
            candidates.append((area, (x1, y1, x2, y2)))

    if not candidates:
        return None
    return max(candidates, key=lambda t: t[0])[1]


def main():
    args = parse_args()

    # Model
    print(f"[INFO] Loading model: {args.model}")
    model = YOLO(args.model)

    # Source
    src = args.source
    cap: cv2.VideoCapture
    if src.isdigit():
        cap = cv2.VideoCapture(int(src))
    else:
        cap = cv2.VideoCapture(src)
    if not cap.isOpened():
        print(f"[ERR] Cannot open source: {src}")
        sys.exit(1)

    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) or 1280
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 720
    fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0

    # Video writer (optional)
    writer = None
    if args.save_video:
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(args.save_video, fourcc, fps, (width, height))
        if not writer.isOpened():
            print("[WARN] Could not open video writer; disabling save_video.")
            writer = None

    # OCR
    ocr = OCR(use_gpu=args.gpu)

    # CSV
    os.makedirs(os.path.dirname(args.out_csv) or ".", exist_ok=True)
    csvf = open(args.out_csv, "w", newline="")
    w = csv.writer(csvf)
    w.writerow(["frame_idx","timestamp","latency_ms","bbox","raw_text","pred_date"])

    print("[INFO] Press 'q' to quit window (if --show).")
    frame_idx = 0
    lat_samples = []

    # warm-up (helps stabilize latency)
    warm = np.zeros((height, width, 3), dtype=np.uint8)
    for _ in range(2):
        _ = model.predict(warm, conf=args.conf, iou=args.iou, verbose=False)

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        t0 = time.time()
        result = model.predict(frame, conf=args.conf, iou=args.iou, verbose=False)[0]
        dt_ms = (time.time() - t0) * 1000.0
        lat_samples.append(dt_ms)

        # ROI selection
        roi_box = pick_roi_from_yolo(result, args.target_class, frame.shape[1], frame.shape[0])
        raw_text, pred_norm = "", ""
        if roi_box is not None:
            x1,y1,x2,y2 = roi_box
            roi = frame[y1:y2, x1:x2]
            raw_text = ocr.run(roi)
            pred_norm = normalize_date(raw_text)

            # draw ROI & text
            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
            label = f"ROI date: {pred_norm or 'N/A'}"
            cv2.putText(frame, label, (x1, max(0,y1-7)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

        # draw all dets for context
        for b in result.boxes:
            x1,y1,x2,y2 = [int(v) for v in b.xyxy[0].tolist()]
            cls_id = int(b.cls[0].item()) if b.cls is not None else -1
            conf   = float(b.conf[0].item()) if b.conf is not None else 0.0
            cv2.rectangle(frame, (x1,y1), (x2,y2), (255,0,0), 1)
            cv2.putText(frame, f"{cls_id}:{conf:.2f}", (x1, y1-3),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,0), 1)

        # latency overlay
        cv2.putText(frame, f"YOLOv10 latency: {dt_ms:.1f} ms", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

        # write CSV row
        ts = time.time()
        bbox_str = "" if roi_box is None else f"{roi_box[0]},{roi_box[1]},{roi_box[2]},{roi_box[3]}"
        w.writerow([frame_idx, f"{ts:.3f}", f"{dt_ms:.1f}", bbox_str, raw_text, pred_norm])

        # display & save
        if writer is not None:
            writer.write(frame)
        if args.show:
            cv2.imshow("YOLOv10 + ROI OCR", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        frame_idx += 1
        if args.max_frames and frame_idx >= args.max_frames:
            break

    cap.release()
    if writer is not None:
        writer.release()
    csvf.close()
    if args.show:
        cv2.destroyAllWindows()

    # summary
    if lat_samples:
        mean_lat = sum(lat_samples)/len(lat_samples)
        p50 = np.percentile(lat_samples, 50)
        p95 = np.percentile(lat_samples, 95)
        print(f"[SUMMARY] Frames: {len(lat_samples)} | mean {mean_lat:.1f} ms | p50 {p50:.1f} | p95 {p95:.1f}")
    print(f"[OK] CSV saved: {args.out_csv}")
    if args.save_video:
        print(f"[OK] Annotated video: {args.save_video}")
    print("[DONE]")


if __name__ == "__main__":
    main()


usage: colab_kernel_launcher.py [-h] --model MODEL --source SOURCE
                                [--out_csv OUT_CSV] [--save_video SAVE_VIDEO]
                                [--conf CONF] [--iou IOU]
                                [--target_class TARGET_CLASS] [--show]
                                [--max_frames MAX_FRAMES] [--gpu]
colab_kernel_launcher.py: error: the following arguments are required: --model, --source
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/lib/python3.12/argparse.py", line 1943, in _parse_known_args2
    namespace, args = self._parse_known_args(args, namespace, intermixed)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 2230, in _parse_known_args
    raise ArgumentError(None, _('the following arguments are required: %s') %
argparse.ArgumentError: the following arguments are required: --model, --source

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipython-input-70797762.py", line 288, in <cell line: 0>
    main()
  File "/tmp/ipython-input-70797762.py", line 167, in main
    args = parse_args()
           ^^^^^^^^^^^^
  File "/tmp/ipython-input-70797762.py", line 55, in parse

TypeError: object of type 'NoneType' has no len()